In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
df = pd.read_csv('datos/datos_limpios.csv')

In [ ]:
# Nombres de las estaciones en orden
stations = [
    "SURESTE", "NORESTE", "CENTRO", "NOROESTE", "SUROESTE",
    "NOROESTE 2", "NORTE", "NORESTE2", "SUROESTE2", "SURESTE2",
    "SURESTE 3", "SUR", "NORTE 2", "NORESTE 3", "NOROESTE 3"
]

block_size = 15  # número de variables por estación

# Diccionario para guardar los DataFrames
dfs_estaciones = {}

for i, station in enumerate(stations):
    start = 1 + i * block_size
    end = start + block_size
    cols = ["Date"] + df.columns[start:end].tolist()
    dfs_estaciones[station] = df[cols].copy()



In [ ]:
from utils import create_variable_boxplot
# Crear boxplots para todas las variables (1-15)
print("Creando boxplots para todas las variables...")
print("=" * 70)

for var_idx in range(1, 16):  # Variables del 1 al 15 (0 es Date)
    print(f"\nVariable {var_idx}:")
    create_variable_boxplot(var_idx, dfs_estaciones)
    print("\n" + "=" * 70)

In [ ]:
from utils import create_correlation_heatmap
# Crear mapas de correlación para todas las variables (1-15)
print("\n" + "=" * 70)
print("Creando mapas de correlación para TODAS las variables...")
print("=" * 70)

for var_idx in range(1, 16):  # Variables del 1 al 15 (0 es Date)
    print(f"\nVariable {var_idx}:")
    create_correlation_heatmap(var_idx, dfs_estaciones)
    print("\n" + "=" * 70)

In [ ]:
from utils import correct_outliers, generate_correction_report, validate_corrections

# Aplicar la corrección inteligente de outliers con límites físicos y z-score móvil
print("🔧 INICIANDO CORRECCIÓN INTELIGENTE DE OUTLIERS")
print("=" * 60)
print("✨ Características:")
print("   • Límites físicos realistas para cada tipo de variable")
print("   • Detección por z-score móvil (ventana de 24 horas)")
print("   • Preservación de eventos ambientales reales")
print("   • Corrección específica por tipo de contaminante")

# Realizar correcciones con parámetros optimizados
dfs_estaciones_clean, correction_summary = correct_outliers(
    dfs_estaciones, 
    method='rolling_zscore',    # Método de z-score móvil
    window_size=24,             # Ventana de 24 horas
    z_threshold=3.5,           # Umbral conservador para preservar eventos reales
    verbose=True
)

# Generar reporte detallado
generate_correction_report(correction_summary, dfs_estaciones, dfs_estaciones_clean)

# Validar correcciones
validate_corrections(dfs_estaciones, dfs_estaciones_clean)

print(f"\n✨ PROCESO COMPLETADO CON ÉXITO")
print("📦 Datasets disponibles:")
print("   • dfs_estaciones       : Datos originales")
print("   • dfs_estaciones_clean : Datos limpios (recomendado para análisis)")
print("\n🎯 Listo para análisis estadístico avanzado!")

In [ ]:
from utils import compare_datasets_statistics, create_before_after_plots

print("📊" * 30)
print("ANÁLISIS COMPARATIVO: ORIGINAL vs LIMPIO")
print("📊" * 30)

# Ejemplo 1: Comparación estadística para la variable 1
print("\n1️⃣ COMPARACIÓN ESTADÍSTICA - Variable 1:")
try:
    stats_comparison_var1 = compare_datasets_statistics(dfs_estaciones, dfs_estaciones_clean, 1)
    print(f"\n✅ DataFrame de comparación creado con {len(stats_comparison_var1)} estaciones")
except Exception as e:
    print(f"❌ Error en comparación estadística: {e}")

# Ejemplo 2: Visualización antes/después para una estación específica
print(f"\n2️⃣ VISUALIZACIÓN ANTES/DESPUÉS:")
try:
    # Seleccionar la primera estación como ejemplo
    ejemplo_estacion = list(dfs_estaciones.keys())[0]
    print(f"🏭 Mostrando análisis para la estación: {ejemplo_estacion}")
    create_before_after_plots(dfs_estaciones, dfs_estaciones_clean, 1, ejemplo_estacion)
except Exception as e:
    print(f"❌ Error en visualización: {e}")

# Ejemplo 3: Verificación rápida de todas las variables
print(f"\n3️⃣ RESUMEN DE CORRECCIONES POR VARIABLE:")
print("-" * 70)
print("Variable | Outliers Orig | Outliers Final | Mejora | Estado")
print("-" * 70)

total_original_issues = 0
total_final_issues = 0

for var_idx in range(1, 16): 
    orig_issues = 0
    final_issues = 0
    
    for station_name, df_orig in dfs_estaciones.items():
        df_clean = dfs_estaciones_clean[station_name]
        
        if var_idx < len(df_orig.columns):
            # Contar valores problemáticos (negativos + extremos)
            orig_data = df_orig.iloc[:, var_idx]
            clean_data = df_clean.iloc[:, var_idx]
            
            # Problemas originales: negativos + valores muy extremos
            orig_problems = (orig_data < 0).sum()
            if not orig_data.empty:
                q99 = orig_data.quantile(0.99)
                q01 = orig_data.quantile(0.01)
                orig_problems += ((orig_data > q99 * 5) | (orig_data < q01 * 5)).sum()
            
            # Problemas finales: similar análisis
            final_problems = (clean_data < 0).sum()
            if not clean_data.empty:
                q99 = clean_data.quantile(0.99) 
                q01 = clean_data.quantile(0.01)
                final_problems += ((clean_data > q99 * 5) | (clean_data < q01 * 5)).sum()
            
            orig_issues += orig_problems
            final_issues += final_problems
    
    total_original_issues += orig_issues
    total_final_issues += final_issues
    
    # Calcular mejora
    improvement = ((orig_issues - final_issues) / max(orig_issues, 1)) * 100
    
    # Estado
    if orig_issues == 0:
        status = "✨ Perfecto"
    elif final_issues == 0:
        status = "🎯 Resuelto"
    elif improvement > 80:
        status = "✅ Excelente"
    elif improvement > 50:
        status = "👍 Bueno"
    elif improvement > 20:
        status = "⚠️ Parcial"
    else:
        status = "🔍 Revisar"
    
    print(f"Var {var_idx:2d}   | {orig_issues:8d}    | {final_issues:8d}     | {improvement:5.1f}% | {status}")

print("-" * 70)
improvement_total = ((total_original_issues - total_final_issues) / max(total_original_issues, 1)) * 100
print(f"TOTAL    | {total_original_issues:8d}    | {total_final_issues:8d}     | {improvement_total:5.1f}% | 🎯")
print("-" * 70)

print(f"\n🎉 LIMPIEZA COMPLETADA:")
print(f"   • Problemas originales: {total_original_issues}")
print(f"   • Problemas restantes: {total_final_issues}") 
print(f"   • Mejora global: {improvement_total:.1f}%")
print(f"   • Datos listos para análisis estadístico avanzado! 🚀")

In [ ]:
from utils import create_compact_histogram

print("📊" * 25)
print("ANÁLISIS DE NORMALIDAD - DATOS LIMPIOS")
print("📊" * 25)
print("Generando histogramas con pruebas de normalidad usando datos completamente limpios")
print("(Sin outliers, con límites físicos aplicados)")
print("=" * 80)

normality_summary_clean = {}

for var_idx in range(1, 16):
    print(f"\n🔬 Variable {var_idx}:")
    results = create_compact_histogram(var_idx, dfs_estaciones_clean)
    normality_summary_clean[var_idx] = results
    print("\n" + "="*80)

print(f"\n🎯 ANÁLISIS DE NORMALIDAD COMPLETADO")
print(f"   📈 Variables analizadas: 15")
print(f"   🏭 Estaciones procesadas: {len(dfs_estaciones_clean)}")
print(f"   📊 Datos utilizados: Completamente limpios")
print(f"   💾 Resultados guardados en: normality_summary_clean")

In [ ]:
print("📦" * 25)
print("BOXPLOTS FINALES - DATOS COMPLETAMENTE LIMPIOS")
print("📦" * 25)
print("Generando boxplots finales con datos sin outliers y límites físicos aplicados")
print("=" * 70)

for var_idx in range(1, 16):  # Variables del 1 al 15 (0 es Date)
    print(f"\n📊 Variable {var_idx}:")
    create_variable_boxplot(var_idx, dfs_estaciones_clean)
    print("\n" + "=" * 70)

print(f"\n🎯 ANÁLISIS EXPLORATORIO COMPLETADO")
print(f"   📈 Se generaron boxplots para 15 variables")
print(f"   🏭 Datos de {len(dfs_estaciones_clean)} estaciones procesadas")
print(f"   ✨ Todos los gráficos utilizan datos completamente limpios")
print(f"   🚀 Listos para análisis estadístico avanzado!")